[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cursuri-inf/DCTC/blob/main/notebooks/analiza-sablon.ipynb)

# Analiza datelor echipei: șablonul DCTC

Șablonul temelor 1–6 ale proiectului, cu date dintr-un fișier CSV. Notebook-ul citește datele direct din depozitul echipei, le curăță, calculează statistici descriptive, desenează două grafice și le salvează ca PNG pentru raport. Implicit rulează pe datele de exemplu din depozitul disciplinei, care sunt generate, nu reale.

Cum îl folosești:

1. *File > Save a copy in Drive*: lucrezi pe copia ta, șablonul rămâne neschimbat.
2. *Runtime > Run all*: șablonul rulează pe datele de exemplu în mai puțin de un minut.
3. Schimbi celula de parametri (secțiunea 1) și textele marcate „De completat”; codul de bază rămâne cum este.
4. Înainte de a salva în GitHub: *Runtime > Restart session and run all*, ca rezultatele salvate să fie ale unei rulări de la zero.

În notebook nu intră parole, chei sau căi de pe calculatorul tău.

## 1. Parametrii

Singura celulă de cod pe care o modifici la început. `URL_DATE` este adresa raw a fișierului CSV: pe GitHub deschizi fișierul din `data/` și copiezi adresa butonului *Raw*; începe cu `https://raw.githubusercontent.com/`. Numele coloanelor se scriu exact ca în primul rând al fișierului.

In [ ]:
# Parametrii analizei
URL_DATE = "https://raw.githubusercontent.com/cursuri-inf/DCTC/main/date-exemplu/exemplu-serii.csv"
SEPARATOR = ","                  # ";" dacă fișierul folosește punct și virgulă între coloane
COLOANA_TIMP = "an"              # axa orizontală: anul (sau o dată calendaristică)
COLOANA_VALOARE = "valoare"      # coloana analizată, numerică
COLOANA_GRUP = "regiune"         # coloana care desparte seriile (țara, județul); None dacă e o singură serie
GRUPURI = ["Regiunea A", "Regiunea B", "Regiunea C"]  # filtrul: seriile comparate; [] înseamnă toate
PRINCIPAL = "Regiunea A"         # seria despre care vorbește studiul, colorată în grafice
TIMP_MIN, TIMP_MAX = 2000, 2025  # filtrul pe timp, cu capete cu tot
ETICHETA = "Valoarea indicatorului (unități)"  # textul axei cu valorile
FOLDER_REZULTATE = "rezultate"   # unde se salvează PNG-urile
DESCARCA_FIGURI = False          # True: la final, Colab descarcă PNG-urile pe calculator

## 2. Bibliotecile

pandas lucrează cu tabele, matplotlib desenează graficele; în Colab sunt deja instalate. Versiunile afișate ajută pe oricine vrea să reproducă rezultatele.

In [ ]:
from pathlib import Path

import matplotlib
import matplotlib.pyplot as plt
import pandas as pd

CULOARE_PRINCIPAL = "#9B1B30"    # seria principală
CULOARE_REST = "#8A8A8A"         # celelalte serii
print("pandas", pd.__version__, "· matplotlib", matplotlib.__version__)

## 3. Datele

Datele se citesc direct din depozit, prin URL-ul raw: oricine rulează notebook-ul primește aceleași date, fără încărcări manuale în sesiunea Colab.

In [ ]:
date = pd.read_csv(URL_DATE, sep=SEPARATOR)
print(f"{len(date)} rânduri și {date.shape[1]} coloane: {list(date.columns)}")
date.head()

## 4. Curățarea minimă

Coloanele de timp și de valori devin numere (virgula zecimală devine punct), rândurile fără valoare se elimină, se aplică filtrul pe timp și pe serii. Dacă o serie are mai multe valori în același an (de exemplu, date zilnice), se folosește media lor anuală.

In [ ]:
lipsa = [c for c in (COLOANA_TIMP, COLOANA_VALOARE, COLOANA_GRUP) if c and c not in date.columns]
if lipsa:
    raise KeyError(f"Coloanele {lipsa} nu există în fișier. Coloanele disponibile: {list(date.columns)}")

grup = COLOANA_GRUP or "serie"
if not COLOANA_GRUP:
    date[grup] = "toate datele"

timp = pd.to_numeric(date[COLOANA_TIMP], errors="coerce")
if timp.isna().all():  # o dată calendaristică: se păstrează anul
    timp = pd.to_datetime(date[COLOANA_TIMP], errors="coerce").dt.year
date[COLOANA_TIMP] = timp
date[COLOANA_VALOARE] = pd.to_numeric(
    date[COLOANA_VALOARE].astype(str).str.replace(",", ".", regex=False), errors="coerce")

inainte = len(date)
curat = date.dropna(subset=[COLOANA_TIMP, COLOANA_VALOARE])
curat = curat[(curat[COLOANA_TIMP] >= TIMP_MIN) & (curat[COLOANA_TIMP] <= TIMP_MAX)]
if GRUPURI and COLOANA_GRUP:
    curat = curat[curat[grup].isin(GRUPURI)]
curat = curat.groupby([grup, COLOANA_TIMP], as_index=False)[COLOANA_VALOARE].mean()
curat[COLOANA_TIMP] = curat[COLOANA_TIMP].astype(int)
print(f"Au rămas {len(curat)} rânduri din {inainte}; seriile: {sorted(curat[grup].astype(str).unique())}")

## 5. Statistici descriptive

Pentru fiecare serie, cu valorile $x_1, \dots, x_n$, media și deviația standard sunt

$$\bar{x} = \frac{1}{n}\sum_{i=1}^{n} x_i, \qquad s = \sqrt{\frac{1}{n-1}\sum_{i=1}^{n}\left(x_i - \bar{x}\right)^2},$$

iar variația relativă de la primul la ultimul an este $\Delta = \dfrac{x_{\mathrm{final}} - x_{\mathrm{inițial}}}{x_{\mathrm{inițial}}} \cdot 100\%$.

In [ ]:
def variatie(serie):
    """Variația relativă, în procente, de la prima la ultima valoare a seriei."""
    serie = serie.reset_index(drop=True)
    return (serie.iloc[-1] - serie.iloc[0]) / serie.iloc[0] * 100

statistici = curat.groupby(grup)[COLOANA_VALOARE].agg(["count", "mean", "std", "min", "max", variatie])
statistici.columns = ["n", "media", "deviația standard", "minimul", "maximul", "variația (%)"]
statistici.round(2)

## 6. Graficul de evoluție

O linie pentru fiecare serie. Seria principală (parametrul `PRINCIPAL`) este colorată, celelalte sunt gri; numele seriilor stau la capătul liniilor, fără legendă.

In [ ]:
Path(FOLDER_REZULTATE).mkdir(exist_ok=True)
fig, ax = plt.subplots(figsize=(8, 4.5))
for nume, serie in curat.groupby(grup):
    este_principal = str(nume) == str(PRINCIPAL)
    culoare = CULOARE_PRINCIPAL if este_principal else CULOARE_REST
    ax.plot(serie[COLOANA_TIMP], serie[COLOANA_VALOARE], color=culoare,
            linewidth=2.4 if este_principal else 1.3, zorder=3 if este_principal else 2)
    ax.annotate(str(nume), (serie[COLOANA_TIMP].iloc[-1], serie[COLOANA_VALOARE].iloc[-1]),
                xytext=(5, 0), textcoords="offset points", va="center", color=culoare)
ax.set_xlabel(COLOANA_TIMP)
ax.set_ylabel(ETICHETA)
ax.set_title("Evoluția în timp", loc="left")
ax.spines[["top", "right"]].set_visible(False)
fig.tight_layout()
fig.savefig(f"{FOLDER_REZULTATE}/evolutie.png", dpi=200)
plt.show()

## 7. Graficul de comparație

Media fiecărei serii, ca bare orizontale ordonate; când există o singură serie, histograma valorilor.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
if curat[grup].nunique() > 1:
    medii = statistici["media"].sort_values()
    culori = [CULOARE_PRINCIPAL if str(g) == str(PRINCIPAL) else CULOARE_REST for g in medii.index]
    ax.barh([str(g) for g in medii.index], medii.values, color=culori)
    ax.set_xlabel(f"Media: {ETICHETA}")
    ax.set_title("Comparația mediilor", loc="left")
else:
    ax.hist(curat[COLOANA_VALOARE], bins=15, color=CULOARE_REST, edgecolor="white")
    ax.set_xlabel(ETICHETA)
    ax.set_ylabel("Numărul de ani")
    ax.set_title("Distribuția valorilor", loc="left")
ax.spines[["top", "right"]].set_visible(False)
fig.tight_layout()
fig.savefig(f"{FOLDER_REZULTATE}/comparatie.png", dpi=200)
plt.show()

## 8. Concluzia

De completat: 3–5 propoziții care răspund la întrebarea studiului, cu valori concrete din tabel și din grafice, și o limită a analizei.

## 9. Figurile pentru raport

PNG-urile sunt în folderul `rezultate` al sesiunii Colab (panoul *Files*, în stânga) și dispar când sesiunea se închide. Le descarci (clic dreapta pe fișier, *Download*, sau `DESCARCA_FIGURI = True`) și le pui în depozitul echipei, în `report/figures/`, printr-un pull request.

In [ ]:
figuri = sorted(Path(FOLDER_REZULTATE).glob("*.png"))
print("Figurile salvate:", [f.name for f in figuri])
if DESCARCA_FIGURI:
    try:
        from google.colab import files
        for f in figuri:
            files.download(str(f))
    except ImportError:
        print("Notebook-ul nu rulează în Colab; figurile sunt în folderul", FOLDER_REZULTATE)